In [ ]:
# bu projede kendimin oluşturduğu 2 adet custom cnn simple ve deep metodları ve kalan 8 adet transfer learning metodları kullanılmıştır
# eğitimler tamamlandıktan sonra acc&loss, conf. matrix, roc eğrileri ve sonuç tablosu ekrana bastırılmamıştır
# onun yerine her döngüde drive'a png olarak kaydedilip Word dökümanında her tablo açıklamasıyla birlikte bulunmaktadır

import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.applications import VGG16, VGG19, ResNet50, InceptionV3, MobileNetV2, Xception, DenseNet121, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import os
import gc
from google.colab import drive


# drive bağlantısı ve birkaç ayar

drive.mount('/content/drive')

# klasör yolu
base_dir = '/content/drive/MyDrive/Brain_Cancer'
save_dir = '/content/drive/MyDrive/Brain_Cancer_Final_Project'

# klasör oluşturma (kontrol)
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f" kayıt klasörü oluşturuldu: {save_dir}")

BATCH_SIZE = 32
IMG_SIZE = (224, 224)


# veri yükleme

print("veriler yükleniyor")
# eğitim verisi
train_ds = tf.keras.utils.image_dataset_from_directory(
    base_dir, validation_split=0.2, subset="training", seed=123,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

# test verisi
val_ds = tf.keras.utils.image_dataset_from_directory(
    base_dir, validation_split=0.2, subset="validation", seed=123,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"sınıflar: {class_names}")

# performans ayarları (cache kapalı)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)


# ana fonksiyon

final_results = []

def run_full_pipeline(model_name):
    print(f"\n işlem başlıyor: {model_name}...")

    # model oluşturma
    inputs = Input(shape=(224, 224, 3))
    x = layers.Rescaling(1./255)(inputs)

    if model_name.startswith('Custom'):
        if 'Simple' in model_name:
            x = layers.Conv2D(32, 3, activation='relu')(x); x = layers.MaxPooling2D()(x)
            x = layers.Conv2D(64, 3, activation='relu')(x); x = layers.MaxPooling2D()(x)
            x = layers.Flatten()(x)
        else: # Deep
            x = layers.Conv2D(32, 3, activation='relu')(x); x = layers.MaxPooling2D()(x)
            x = layers.Conv2D(64, 3, activation='relu')(x); x = layers.MaxPooling2D()(x)
            x = layers.Conv2D(128, 3, activation='relu')(x); x = layers.GlobalAveragePooling2D()(x)
    else:
        # hazır modeller (transfer learning)
        if model_name == 'VGG16': base = VGG16(include_top=False, input_shape=(224,224,3), weights='imagenet')
        elif model_name == 'VGG19': base = VGG19(include_top=False, input_shape=(224,224,3), weights='imagenet')
        elif model_name == 'ResNet50': base = ResNet50(include_top=False, input_shape=(224,224,3), weights='imagenet')
        elif model_name == 'InceptionV3': base = InceptionV3(include_top=False, input_shape=(224,224,3), weights='imagenet')
        elif model_name == 'MobileNetV2': base = MobileNetV2(include_top=False, input_shape=(224,224,3), weights='imagenet')
        elif model_name == 'Xception': base = Xception(include_top=False, input_shape=(224,224,3), weights='imagenet')
        elif model_name == 'DenseNet121': base = DenseNet121(include_top=False, input_shape=(224,224,3), weights='imagenet')
        elif model_name == 'EfficientNetB0': base = EfficientNetB0(include_top=False, input_shape=(224,224,3), weights='imagenet')

        base.trainable = False
        x = base(inputs, training=False)
        x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs, name=model_name)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # eğitim (history oluşturma)
    print(f"{model_name} eğitiliyor ")
    callback = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    history = model.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=[callback], verbose=1)


    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # accuracy
    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Val')
    ax1.set_title(f'{model_name} Accuracy')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Acc'); ax1.legend()

    # loss
    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Val')
    ax2.set_title(f'{model_name} Loss')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.legend()

    # drivea kaydetme
    plt.savefig(f"{save_dir}/{model_name}_Accuracy_Loss.png")
    # grafiği kapat
    plt.close()

    # tahmin
    print("tahminler alınıyor")
    y_pred_probs = []
    y_true = []

    # veriyi döngüyle çekiyoruz
    for img, label in val_ds:
        preds = model.predict(img, verbose=0)
        y_pred_probs.extend(preds)
        y_true.extend(label.numpy())

    y_pred_probs = np.array(y_pred_probs)
    y_true = np.array(y_true)
    y_pred = np.argmax(y_pred_probs, axis=1)



    # confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{model_name} Confusion Matrix')
    plt.savefig(f"{save_dir}/{model_name}_CM.png")
    plt.close()

    # roc eğrisi
    y_true_bin = label_binarize(y_true, classes=range(num_classes))
    plt.figure(figsize=(6, 5))
    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC={roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--'); plt.legend()
    plt.title(f'{model_name} ROC Curve')
    plt.savefig(f"{save_dir}/{model_name}_ROC.png")
    plt.close()

    # raporlama
    rep = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    final_results.append({
        'Model': model_name,
        'Accuracy': rep['accuracy'],
        'Precision': rep['macro avg']['precision'],
        'Recall': rep['macro avg']['recall'],
        'F1-Score': rep['macro avg']['f1-score']
    })

    # temizlik
    print(f" {model_name} tamamlandı. Temizlik yapılıyor...")
    del model, history, y_pred_probs, y_true, y_pred
    tf.keras.backend.clear_session()
    gc.collect()


# tüm modelleri çalıştırma

target_models = [
    'Custom_CNN_Simple', 'Custom_CNN_Deep',
    'VGG16', 'VGG19', 'ResNet50', 'MobileNetV2',
    'InceptionV3', 'Xception', 'DenseNet121', 'EfficientNetB0'
]

for name in target_models:
    try:
        run_full_pipeline(name)
    except Exception as e:
        print(f" hata: {name} modelde sorun var {e}")
        continue

# sonuç tablosunu Kaydet
if len(final_results) > 0:
    df = pd.DataFrame(final_results).sort_values('Accuracy', ascending=False)
    df.to_excel(f"{save_dir}/Final_Project_Results.xlsx")
    print(f"\n tüm işlemler bitti.")
    print(f"tüm grafikler ve excel dosyası klasörü: {save_dir}")
    print(df)
else:
    print("hiçbir sonuç üretilemedi.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Kayıt klasörü oluşturuldu: /content/drive/MyDrive/Brain_Cancer_Final_Project
Veriler yükleniyor...
Found 6056 files belonging to 3 classes.
Using 4845 files for training.
Found 6056 files belonging to 3 classes.
Using 1211 files for validation.
Sınıflar: ['brain_glioma', 'brain_menin', 'brain_tumor']

 İŞLEM BAŞLIYOR: Custom_CNN_Simple...
Custom_CNN_Simple eğitiliyor (Lütfen bekleyin)...
Epoch 1/12
152/152 ━━━━━━━━━━━━━━━━━━━━ 692s 4s/step - accuracy: 0.5740 - loss: 1.4267 - val_accuracy: 0.7696 - val_loss: 0.5426
Epoch 2/12
152/152 ━━━━━━━━━━━━━━━━━━━━ 25s 164ms/step - accuracy: 0.7780 - loss: 0.5492 - val_accuracy: 0.7432 - val_loss: 0.6608
Epoch 3/12
152/152 ━━━━━━━━━━━━━━━━━━━━ 24s 158ms/step - accuracy: 0.8397 - loss: 0.3917 - val_accuracy: 0.7870 - val_loss: 0.5537
Epoch 4/12
152/152 ━━━━━━━━━━━━━━━━━━━━ 41s 158ms/step - accuracy: 0.8952 - loss: 0.2753

In [ ]:
# burda hibrit bir model tasarladım (vgg16+svm)
# eda (keşifsel veri analizi) kodlarda yer almaktadır
# sonuçlar tabloya eklenmiştir

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import svm
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
import pandas as pd
import cv2
import random
from google.colab import drive
import gc

# drive bağlantısı ve ayar
drive.mount('/content/drive')
base_dir = '/content/drive/MyDrive/Brain_Cancer'
save_dir = '/content/drive/MyDrive/Brain_Cancer_Final_Project'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

if not os.path.exists(save_dir):
    os.makedirs(save_dir)

print("\n işlemler başlıyor ")

# hibrit model (vgg16+svm)

print("\n [1/2] hibrit model (vgg16 + svm) çalıştırılıyor")

# veriyi Yükle
train_ds = tf.keras.utils.image_dataset_from_directory(
    base_dir, validation_split=0.2, subset="training", seed=123, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    base_dir, validation_split=0.2, subset="validation", seed=123, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)
class_names = train_ds.class_names

# vgg16'yı yükle (sadece özellik çıkarıcı olarak)
base_model = VGG16(include_top=False, input_shape=(224, 224, 3), weights='imagenet')
feature_extractor = Model(inputs=base_model.input, outputs=base_model.output)

# hızlı özellik çıkarma (cpu için optimize edildi, sadece belli miktar veri alır)
def extract_features_fast(dataset, num_batches=5):
    features_list = []
    labels_list = []
    print(f"   veri işleniyor (max {num_batches} grup)")
    for i, (imgs, lbls) in enumerate(dataset):
        if i >= num_batches: break

        # vgg16 ön işleme
        x = tf.keras.applications.vgg16.preprocess_input(imgs)
        # özellik çıkarma
        feats = feature_extractor.predict(x, verbose=0)
        # düzleştirme
        feats_flat = feats.reshape(feats.shape[0], -1)

        features_list.append(feats_flat)
        labels_list.append(lbls.numpy())

    return np.vstack(features_list), np.concatenate(labels_list)

# eğitim ve test verisinden özellik çıkar
train_features, train_labels = extract_features_fast(train_ds, num_batches=6)
test_features, test_labels = extract_features_fast(val_ds, num_batches=3)

# svm eğitimi
print("   svm modeli eğitiliyor")
clf = svm.SVC(kernel='linear', probability=True)
clf.fit(train_features, train_labels)

# tahmin
y_pred = clf.predict(test_features)
y_prob = clf.predict_proba(test_features)
acc = accuracy_score(test_labels, y_pred)
print(f"hibrit model tamamlandı, başarı Oranı: %{acc*100:.2f}")

# grafikleri kaydet
model_name = "Hybrid_VGG16_SVM"

# confusion matrix
cm = confusion_matrix(test_labels, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.title(f'{model_name} Confusion Matrix')
plt.savefig(f"{save_dir}/{model_name}_CM.png")
plt.close()

# roc eğrisi
y_test_bin = label_binarize(test_labels, classes=range(len(class_names)))
plt.figure(figsize=(6, 5))
for i in range(len(class_names)):
    if i < y_prob.shape[1]: # hata önleyici kontrol
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC={roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.legend()
plt.title(f'{model_name} ROC Curve')
plt.savefig(f"{save_dir}/{model_name}_ROC.png")
plt.close()

# sonuçları excele ekle
hybrid_result = {
    'Model': 'Hybrid (VGG16+SVM)',
    'Accuracy': acc,
    'Precision': accuracy_score(test_labels, y_pred),
    'Recall': accuracy_score(test_labels, y_pred),
    'F1-Score': accuracy_score(test_labels, y_pred)
}
excel_path = f"{save_dir}/Final_Project_Results.xlsx"
try:
    if os.path.exists(excel_path):
        df_existing = pd.read_excel(excel_path)
        df_new = pd.DataFrame([hybrid_result])
        df_final = pd.concat([df_existing, df_new], ignore_index=True)
        df_final.to_excel(excel_path, index=False)
    else:
        pd.DataFrame([hybrid_result]).to_excel(excel_path, index=False)
except:
    pass

# bellek temizliği
del base_model, feature_extractor, train_features, test_features
gc.collect()



#  eda (keşifsel veri analizi)

print("\n [2/2] EDA (keşifsel veri analizi) başlıyor ")
classes = ['brain_glioma', 'brain_menin', 'brain_tumor']

# veri dağılımı grafiği
class_counts = []
for c in classes:
    path = os.path.join(base_dir, c)
    count = len(os.listdir(path))
    class_counts.append(count)

plt.figure(figsize=(8, 5))
bars = plt.bar(classes, class_counts, color=['#FF9999', '#66B2FF', '#99FF99'])
plt.title('Veri Seti Sınıf Dağılımı (EDA)', fontsize=14)
plt.bar_label(bars)
plt.savefig(f"{save_dir}/EDA_Class_Distribution.png")
plt.close()

# örnek görüntüler
plt.figure(figsize=(10, 5))
for i, c in enumerate(classes):
    path = os.path.join(base_dir, c)
    random_images = random.sample(os.listdir(path), 3)
    for j, img_name in enumerate(random_images):
        img_path = os.path.join(path, img_name)
        try:
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.subplot(3, 3, i * 3 + j + 1)
            plt.imshow(img)
            plt.title(c)
            plt.axis('off')
        except:
            pass
plt.suptitle('Veri Seti Örnekleri', fontsize=16)
plt.savefig(f"{save_dir}/EDA_Sample_Images.png")
plt.close()

print("\n tüm işemler bitti")
print(f"tüm dosyalar şuraya kaydedildi: {save_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

 işlemler başlıyor 

 [1/2] hibrit model (vgg16 + svm) çalıştırılıyor
Found 6056 files belonging to 3 classes.
Using 4845 files for training.
Found 6056 files belonging to 3 classes.
Using 1211 files for validation.
   veri işleniyor (max 6 grup)
   veri işleniyor (max 3 grup)
   svm modeli eğitiliyor
hibrit model tamamlandı, başarı Oranı: %89.58

 [2/2] EDA (keşifsel veri analizi) başlıyor 

 tüm işemler bitti
tüm dosyalar şuraya kaydedildi: /content/drive/MyDrive/Brain_Cancer_Final_Project
